In [ ]:
import tkinter as tk
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import os
import cv2
import numpy as np
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend


def analyze_correlations(k,pt_fixe, N=1000, display=True):
    """
    Calcule et affiche les matrices de corrélations cryptographiques.
    
    Paramètres :
        k (str): La clé de 8 bits utilisée pour l'expérience 1.
        pt_fixe (str) : Texte clair arbitraire pour l'expérience 2.
        N (int): Le nombre d'échantillons (textes/clés) à générer.
        display (bool): Si True, affiche les matrices avec Matplotlib.
        
        
    Retourne :
        tuple: (corr_in_out, corr_key_out) sous forme de tableaux Numpy 8x8.
    """
    

    # --- 1. Expérience 1 : Matrice Entrées / Sorties (Clé fixe) ---
    inputs_bits = []
    outputs_bits_exp1 = []

    for _ in range(N):
        pt = "".join(str(random.randint(0, 1)) for _ in range(8))
        ct = encrypt(pt, k, rounds=4, verbose=False)
        inputs_bits.append([int(b) for b in pt])
        outputs_bits_exp1.append([int(b) for b in ct])

    inputs_matrix = np.array(inputs_bits)
    outputs_matrix_exp1 = np.array(outputs_bits_exp1)

    corr_in_out = np.zeros((8, 8))
    for i in range(8):
        for j in range(8):
            corr_matrix = np.corrcoef(inputs_matrix[:, i], outputs_matrix_exp1[:, j])
            corr = 0.0 if np.isnan(corr_matrix[0, 1]) else abs(corr_matrix[0, 1])
            corr_in_out[i, j] = corr

    # --- 2. Expérience 2 : Matrice Clés / Sorties (Texte fixe) ---
    keys_bits = []
    outputs_bits_exp2 = []

    for _ in range(N):
        key_rand = "".join(str(random.randint(0, 1)) for _ in range(8))
        ct = encrypt(pt_fixe, key_rand, rounds=4, verbose=False)
        keys_bits.append([int(b) for b in key_rand])
        outputs_bits_exp2.append([int(b) for b in ct])

    keys_matrix = np.array(keys_bits)
    outputs_matrix_exp2 = np.array(outputs_bits_exp2)

    corr_key_out = np.zeros((8, 8))
    for i in range(8):
        for j in range(8):
            corr_matrix = np.corrcoef(keys_matrix[:, i], outputs_matrix_exp2[:, j])
            corr = 0.0 if np.isnan(corr_matrix[0, 1]) else abs(corr_matrix[0, 1])
            corr_key_out[i, j] = corr

    # --- 3. Affichage Graphique ---
    if display:
        colors = [
            (0.00, "#28a745"),  # 0.00 : Vert (Fluctuation normale)
            (0.10, "#fd7e14"),  # 0.10 : Orange (Zone suspecte)
            (0.15, "#dc3545"),  # 0.15 : Rouge (Faille statistique avérée)
            (0.99, "#dc3545"),  # 0.15 à 0.99 : Reste rouge
            (1.00, "#000000")   # 1.00 : Noir (Corrélation absolue)
        ]
        cmap_crypto = LinearSegmentedColormap.from_list("crypto_alert", colors)

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        def plot_matrix(ax, matrix, title, ylabel):
            cax = ax.imshow(matrix, cmap=cmap_crypto, vmin=0, vmax=1)
            ax.set_title(title, pad=15, fontweight='bold')
            ax.set_xticks(range(8))
            ax.set_yticks(range(8))
            ax.set_xticklabels([f"Out {i}" for i in range(8)])
            ax.set_yticklabels([f"{ylabel} {i}" for i in range(8)])
            
            for i in range(8):
                for j in range(8):
                    text_color = "white" if matrix[i, j] >= 0.15 else "black"
                    ax.text(j, i, f"{matrix[i, j]:.2f}", ha='center', va='center', color=text_color, fontsize=9, fontweight='bold')
            return cax

        # Dessin des matrices
        plot_matrix(axes[0], corr_in_out, f"Corrélation Entrées / Sorties\n(Clé fixe : {k})", "In")
        im = plot_matrix(axes[1], corr_key_out, f"Corrélation Clés / Sorties\n(Message fixe : {pt_fixe})", "Key")

        cbar = fig.colorbar(im, ax=axes.ravel().tolist(), orientation='horizontal', fraction=0.05, pad=0.1)
        cbar.set_label("0 = Sécurisé (Vert)   |   0.10 = Suspect (Orange)   |   >0.15 = Vulnérable (Rouge)   |   1 = Cassé (Noir)", fontweight='bold')

        plt.show()

    return corr_in_out, corr_key_out


class avalanche_visualizer:
    def __init__(self, master, pt1, pt2, key):
        self.master = master
        self.master.title("Effet Avalanche Interactif")
        self.master.configure(padx=20, pady=20)
        
        # La clé devient modifiable comme les clairs
        self.key = list(key)
        self.pt1 = list(pt1)
        self.pt2 = list(pt2)
        
        # Mémoire des états précédents pour la comparaison
        self.prev_ct1_str = None
        self.prev_ct2_str = None
        
        self.color_0 = "white"
        self.color_1 = "black"

        self.create_widgets()
        self.update_ciphertexts()

    def create_widgets(self):
        # --- En-tête : La Clé Cliquable ---
        # Utilisation d'un Frame pour regrouper et centrer la clé
        key_frame = tk.Frame(self.master)
        key_frame.grid(row=0, column=0, columnspan=19, pady=(0, 20))
        
        tk.Label(key_frame, text="Clé : ", font=("Calibri", 14, "bold")).pack(side=tk.LEFT, padx=5)
        
        self.key_buttons = []
        for i in range(8):
            btn = tk.Button(key_frame, width=2, height=1, command=lambda idx=i: self.toggle_key_bit(idx))
            btn.pack(side=tk.LEFT, padx=2)
            self.key_buttons.append(btn)

        # --- Ligne Titres ---
        tk.Label(self.master, text="Texte 1", font=("Calibri", 12, "bold")).grid(row=1, column=1, columnspan=8)
        tk.Label(self.master, text="Texte 2", font=("Calibri", 12, "bold")).grid(row=1, column=10, columnspan=8)

        # Séparateur central
        tk.Label(self.master, text=" | ", font=("Calibri", 14)).grid(row=2, column=9, rowspan=3)

        # --- Ligne 1 : Textes Clairs (Boutons cliquables) ---
        tk.Label(self.master, text="Clair : ", font=("Calibri", 12)).grid(row=2, column=0, sticky="e")
        
        self.pt1_buttons = []
        for i in range(8):
            btn = tk.Button(self.master, width=2, height=1, command=lambda idx=i: self.toggle_bit(1, idx))
            btn.grid(row=2, column=i+1, padx=2, pady=5)
            self.pt1_buttons.append(btn)

        self.pt2_buttons = []
        for i in range(8):
            btn = tk.Button(self.master, width=2, height=1, command=lambda idx=i: self.toggle_bit(2, idx))
            btn.grid(row=2, column=i+10, padx=2, pady=5)
            self.pt2_buttons.append(btn)

        # --- Ligne 2 : Textes Chiffrés (Affichage statique) ---
        tk.Label(self.master, text="Chiffré : ", font=("Arial", 12)).grid(row=3, column=0, sticky="e")
        
        self.ct1_labels = []
        for i in range(8):
            lbl = tk.Label(self.master, width=3, height=1, relief="solid")
            lbl.grid(row=3, column=i+1, padx=2, pady=5)
            self.ct1_labels.append(lbl)

        self.ct2_labels = []
        for i in range(8):
            lbl = tk.Label(self.master, width=3, height=1, relief="solid")
            lbl.grid(row=3, column=i+10, padx=2, pady=5)
            self.ct2_labels.append(lbl)

        # --- Ligne 3 : Indicateurs de changement ---
        tk.Label(self.master, text="Modifié : ", font=("Calibri", 10)).grid(row=4, column=0, sticky="e")
        
        self.ct1_diff_labels = []
        for i in range(8):
            lbl = tk.Label(self.master, text="", fg="red", font=("Calibri", 10))
            lbl.grid(row=4, column=i+1, pady=2)
            self.ct1_diff_labels.append(lbl)

        self.ct2_diff_labels = []
        for i in range(8):
            lbl = tk.Label(self.master, text="", fg="red", font=("Calibri", 10))
            lbl.grid(row=4, column=i+10, pady=2)
            self.ct2_diff_labels.append(lbl)

        self.update_ui_colors()

    def toggle_key_bit(self, bit_idx):
        """ Inverse le bit cliqué sur la clé """
        self.key[bit_idx] = '1' if self.key[bit_idx] == '0' else '0'
        self.update_ui_colors()
        self.update_ciphertexts()

    def toggle_bit(self, pt_id, bit_idx):
        """ Inverse le bit cliqué sur les textes clairs """
        if pt_id == 1:
            self.pt1[bit_idx] = '1' if self.pt1[bit_idx] == '0' else '0'
        else:
            self.pt2[bit_idx] = '1' if self.pt2[bit_idx] == '0' else '0'
        
        self.update_ui_colors()
        self.update_ciphertexts()

    def update_ui_colors(self):
        """ Met à jour la couleur de TOUS les boutons cliquables (Clés + Textes) """
        for i, bit in enumerate(self.key):
            color = self.color_1 if bit == '1' else self.color_0
            self.key_buttons[i].configure(bg=color, activebackground=color)

        for i, bit in enumerate(self.pt1):
            color = self.color_1 if bit == '1' else self.color_0
            self.pt1_buttons[i].configure(bg=color, activebackground=color)
            
        for i, bit in enumerate(self.pt2):
            color = self.color_1 if bit == '1' else self.color_0
            self.pt2_buttons[i].configure(bg=color, activebackground=color)

    def update_ciphertexts(self):
        pt1_str = "".join(self.pt1)
        pt2_str = "".join(self.pt2)
        key_str = "".join(self.key)
        
        ct1_str = encrypt(pt1_str, key_str, rounds=4, verbose=False)
        ct2_str = encrypt(pt2_str, key_str, rounds=4, verbose=False)
        
        # Mise à jour et comparaison pour le Texte 1
        for i, bit in enumerate(ct1_str):
            color = self.color_1 if bit == '1' else self.color_0
            self.ct1_labels[i].configure(bg=color)
            
            if self.prev_ct1_str is not None:
                if bit != self.prev_ct1_str[i]:
                    self.ct1_diff_labels[i].configure(text="🔴")
                else:
                    self.ct1_diff_labels[i].configure(text="")
                    
        # Mise à jour et comparaison pour le Texte 2
        for i, bit in enumerate(ct2_str):
            color = self.color_1 if bit == '1' else self.color_0
            self.ct2_labels[i].configure(bg=color)
            
            if self.prev_ct2_str is not None:
                if bit != self.prev_ct2_str[i]:
                    self.ct2_diff_labels[i].configure(text="🔴")
                else:
                    self.ct2_diff_labels[i].configure(text="")

        # On sauvegarde l'état actuel pour le prochain clic
        self.prev_ct1_str = ct1_str
        self.prev_ct2_str = ct2_str





def encrypt_image_ecb(image_path, output_path, key):
    """
    Chiffre une image pixel par pixel avec AES en mode ECB pour démontrer ses failles.
    L'image résultante préservera les motifs de l'image d'origine.
    """
    print(f"--- Chiffrement de {image_path} avec AES-128-ECB ---")

    # 1. Charger l'image avec OpenCV
    # OpenCV charge les images en BGR (Blue, Green, Red) par défaut
    img = cv2.imread(image_path)
    if img is None:
        print(f"Erreur : Impossible de charger l'image {image_path}. Vérifiez le chemin.")
        return

    # Obtenir les dimensions et le nombre de canaux
    height, width, channels = img.shape
    print(f"Dimensions de l'image : {width}x{height} pixels, {channels} canaux.")

    # 2. Aplatir l'image en une séquence d'octets (1D)
    img_bytes = img.tobytes()
    num_bytes = len(img_bytes)
    print(f"Taille des données brutes de l'image : {num_bytes} octets.")

    # 3. Configurer le chiffrement AES-128-ECB
    # AES utilise des blocs de 128 bits (16 octets).
    # Le mode ECB est déterministe et n'utilise pas d'IV.
    cipher = Cipher(algorithms.AES(key), modes.ECB(), backend=default_backend())
    encryptor = cipher.encryptor()

    # 4. Gérer le padding (remplissage)
    # AES-ECB nécessite que la taille des données soit un multiple de 16 octets.
    # Les images ont souvent une taille compatible (width * height * 3),
    # mais il est prudent de vérifier et d'ajouter du padding si nécessaire.
    padding_needed = (16 - (num_bytes % 16)) % 16
    img_bytes_padded = img_bytes + b'\0' * padding_needed
    if padding_needed > 0:
        print(f"Ajout de {padding_needed} octets de padding.")

    # 5. Chiffrer les octets
    # Chaque bloc de 16 octets est chiffré indépendamment.
    encrypted_bytes = encryptor.update(img_bytes_padded) + encryptor.finalize()
    print(f"Taille des données chiffrées : {len(encrypted_bytes)} octets.")

    # 6. Reconstruire l'image à partir des octets chiffrés
    # Enlever le padding pour retrouver la taille d'origine
    if padding_needed > 0:
        encrypted_bytes = encrypted_bytes[:-padding_needed]

    # Convertir les octets chiffrés en un tableau numpy
    encrypted_img_flat = np.frombuffer(encrypted_bytes, dtype=np.uint8)

    # Redimensionner le tableau pour reformer l'image (height, width, channels)
    try:
        encrypted_img = encrypted_img_flat.reshape((height, width, channels))
    except ValueError as e:
        print(f"Erreur lors du redimensionnement de l'image : {e}")
        return

    # 7. Sauvegarder l'image chiffrée
    cv2.imwrite(output_path, encrypted_img)
    print(f"Image chiffrée sauvegardée sous : {output_path}")
    print("Observez comment les motifs et le texte sont toujours visibles.")